<a href="https://colab.research.google.com/github/kshamasahu706-wq/Customer-Churn-Feature-Engineering/blob/main/Deep_Learning_NLP_Text_Classifier_TensorFlow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deep Learning / NLP Text Classifier with TensorFlow
## IMDB Movie Review Sentiment Analysis

**Project type:** Natural Language Processing (NLP) + Deep Learning  
**Framework:** TensorFlow / Keras  
**Task:** Binary sentiment classification — Positive or Negative

### Project objective
Build a neural-network text classifier that learns from movie reviews and predicts whether an unseen review expresses positive or negative sentiment.

### Implementation checklist
- Load and inspect a labeled text dataset.
- Clean and tokenize text using a Keras `TextVectorization` layer.
- Convert tokens into integer sequences and learn word embeddings.
- Build a multi-layer neural network with Batch Normalization and Dropout.
- Train and validate the model using Early Stopping.
- Plot training/validation loss and accuracy curves.
- Evaluate the model on held-out test data.
- Run inference on new sample reviews and show confidence scores.

> **Dataset note:** This notebook uses the IMDB movie-review dataset provided by Keras. The dataset is downloaded automatically in Google Colab the first time it is loaded. It contains labeled movie reviews for a learning demonstration. The model is not intended for high-stakes decisions or as a reliable measure of a person's feelings.

## 1. Import libraries and set random seeds

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import (
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay, accuracy_score
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))
print("Setup complete.")

## 2. Load the IMDB dataset

In [ ]:
# IMDB dataset: 25,000 training reviews and 25,000 test reviews.
# Each review is represented as a sequence of word IDs.
VOCAB_LIMIT = 20000

(train_sequences, y_train), (test_sequences, y_test) = keras.datasets.imdb.load_data(
    num_words=VOCAB_LIMIT
)

print("Training reviews:", len(train_sequences))
print("Test reviews:", len(test_sequences))
print("First training label:", y_train[0], "(0 = negative, 1 = positive)")
print("First encoded review (first 25 IDs):", train_sequences[0][:25])

## 3. Decode reviews back into readable text

In [ ]:
word_index = keras.datasets.imdb.get_word_index()
# Keras IMDB reserves indices 0–3 for padding, start, unknown and unused tokens.
index_to_word = {index + 3: word for word, index in word_index.items()}
index_to_word[0] = "[PAD]"
index_to_word[1] = "[START]"
index_to_word[2] = "[UNK]"
index_to_word[3] = "[UNUSED]"

def decode_review(encoded_review):
    return " ".join(index_to_word.get(int(token), "[UNK]") for token in encoded_review)

train_texts = [decode_review(seq) for seq in train_sequences]
test_texts = [decode_review(seq) for seq in test_sequences]

print("Example review text:")
print(train_texts[0][:1000])
print("\\nLabel:", "Positive" if y_train[0] == 1 else "Negative")

## 4. Create training and validation sets

In [ ]:
# The official IMDB test set is kept untouched for final evaluation.
# Split a validation subset from the training data.
from sklearn.model_selection import train_test_split

X_train, X_val, y_train_split, y_val = train_test_split(
    train_texts,
    np.asarray(y_train),
    test_size=0.20,
    random_state=SEED,
    stratify=y_train
)

y_test = np.asarray(y_test)

print("Training samples:", len(X_train))
print("Validation samples:", len(X_val))
print("Test samples:", len(test_texts))

## 5. Inspect dataset balance

In [ ]:
labels_df = pd.DataFrame({
    "split": ["Train", "Validation", "Test"],
    "negative": [
        int((y_train_split == 0).sum()),
        int((y_val == 0).sum()),
        int((y_test == 0).sum())
    ],
    "positive": [
        int((y_train_split == 1).sum()),
        int((y_val == 1).sum()),
        int((y_test == 1).sum())
    ]
})
display(labels_df)

labels_df.set_index("split")[["negative", "positive"]].plot(
    kind="bar", figsize=(8, 4), rot=0
)
plt.title("Sentiment label counts by data split")
plt.ylabel("Number of reviews")
plt.tight_layout()
plt.show()

## 6. Tokenize and vectorize text

`TextVectorization` standardizes text, splits it into tokens, builds a vocabulary from training data, and converts each review into a fixed-length integer sequence. The layer is adapted using training text only to avoid vocabulary leakage.

In [ ]:
MAX_TOKENS = 20000
SEQUENCE_LENGTH = 250

vectorizer = layers.TextVectorization(
    max_tokens=MAX_TOKENS,
    output_mode="int",
    output_sequence_length=SEQUENCE_LENGTH,
    standardize="lower_and_strip_punctuation"
)

# Adapt only on training reviews.
text_dataset = tf.data.Dataset.from_tensor_slices(X_train).batch(256)
vectorizer.adapt(text_dataset)

vocabulary = vectorizer.get_vocabulary()
print("Vocabulary size:", len(vocabulary))
print("First 20 vocabulary tokens:", vocabulary[:20])

example_vector = vectorizer(tf.constant([X_train[0]]))
print("Vectorized review shape:", example_vector.shape)
print("First 30 token IDs:", example_vector.numpy()[0][:30])

## 7. Build TensorFlow datasets

In [ ]:
BATCH_SIZE = 128
AUTOTUNE = tf.data.AUTOTUNE

def make_dataset(texts, labels, training=False):
    ds = tf.data.Dataset.from_tensor_slices((texts, labels))
    if training:
        ds = ds.shuffle(buffer_size=10000, seed=SEED)
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

train_ds = make_dataset(X_train, y_train_split, training=True)
val_ds = make_dataset(X_val, y_val)
test_ds = make_dataset(test_texts, y_test)

print("TensorFlow datasets are ready.")

## 8. Construct a multi-layer neural network

Architecture:
1. TextVectorization: text to integer token IDs.
2. Embedding: learn a dense vector representation for each token.
3. GlobalAveragePooling1D: summarize sequence information.
4. Dense + Batch Normalization + ReLU: learn non-linear patterns.
5. Dropout: reduce overfitting.
6. Output layer: sigmoid probability for positive sentiment.

Batch Normalization and Dropout are included as requested.

In [ ]:
EMBEDDING_DIM = 128

inputs = keras.Input(shape=(), dtype=tf.string, name="review_text")
x = vectorizer(inputs)
x = layers.Embedding(
    input_dim=MAX_TOKENS,
    output_dim=EMBEDDING_DIM,
    name="word_embedding"
)(x)
x = layers.GlobalAveragePooling1D()(x)

x = layers.Dense(128, use_bias=False)(x)
x = layers.BatchNormalization()(x)
x = layers.Activation("relu")(x)
x = layers.Dropout(0.40)(x)

x = layers.Dense(64, use_bias=False)(x)
x = layers.BatchNormalization()(x)
x = layers.Activation("relu")(x)
x = layers.Dropout(0.30)(x)

outputs = layers.Dense(1, activation="sigmoid", name="positive_probability")(x)

model = keras.Model(inputs, outputs, name="imdb_sentiment_classifier")

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy", keras.metrics.Precision(name="precision"),
             keras.metrics.Recall(name="recall")]
)

model.summary()

## 9. Train with Early Stopping

Early Stopping monitors validation loss. If validation loss does not improve for the configured patience period, training stops and the best model weights are restored. This helps reduce overfitting.

In [ ]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
    verbose=1
)

EPOCHS = 12

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=[early_stopping],
    verbose=1
)

## 10. Plot training and validation convergence curves

In [ ]:
history_df = pd.DataFrame(history.history)
display(history_df.tail())

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(history.history["loss"], label="Training loss")
axes[0].plot(history.history["val_loss"], label="Validation loss")
axes[0].set_title("Loss convergence")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Binary cross-entropy")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history["accuracy"], label="Training accuracy")
axes[1].plot(history.history["val_accuracy"], label="Validation accuracy")
axes[1].set_title("Accuracy convergence")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 11. Evaluate on unseen test data

In [ ]:
test_results = model.evaluate(test_ds, return_dict=True, verbose=1)
print("Test metrics:")
for metric, value in test_results.items():
    print(f"{metric}: {value:.4f}")

In [ ]:
# Generate probabilities and binary predictions for the test set.
test_probabilities = model.predict(test_ds, verbose=1).ravel()
test_predictions = (test_probabilities >= 0.5).astype(int)

print(classification_report(
    y_test,
    test_predictions,
    target_names=["Negative", "Positive"],
    digits=4
))

cm = confusion_matrix(y_test, test_predictions)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Negative", "Positive"]
)
disp.plot(cmap="Blues", values_format="d")
plt.title("Test-set confusion matrix")
plt.show()

## 12. Test on unseen sample inputs

The following reviews are newly written examples, not part of the training or test dataset. The model returns a predicted class and confidence. Confidence is the model's estimated probability, not a guarantee that the prediction is correct.

In [ ]:
sample_reviews = [
    "The movie was fantastic. The acting was brilliant and the story kept me interested.",
    "I hated this film. It was boring, poorly written, and a complete waste of time.",
    "The performances were good, but the plot was slow and the ending was disappointing.",
    "An excellent and emotional film with wonderful music. I would happily watch it again."
]

sample_probabilities = model.predict(
    np.array(sample_reviews, dtype=object),
    verbose=0
).ravel()

sample_rows = []
for review, probability in zip(sample_reviews, sample_probabilities):
    predicted_label = "Positive" if probability >= 0.5 else "Negative"
    confidence = probability if probability >= 0.5 else 1 - probability
    sample_rows.append({
        "Review": review,
        "Predicted sentiment": predicted_label,
        "Positive probability": round(float(probability), 4),
        "Prediction confidence": round(float(confidence), 4)
    })

sample_results = pd.DataFrame(sample_rows)
display(sample_results)

## 13. Create a reusable prediction function

In [ ]:
def predict_sentiment(review):
    """Predict sentiment for one new review and return class probabilities."""
    if not isinstance(review, str) or not review.strip():
        raise ValueError("Please enter a non-empty review.")

    positive_probability = float(model.predict(
        np.array([review], dtype=object), verbose=0
    )[0][0])

    predicted_sentiment = (
        "Positive" if positive_probability >= 0.5 else "Negative"
    )
    confidence = max(positive_probability, 1 - positive_probability)

    return {
        "review": review,
        "predicted_sentiment": predicted_sentiment,
        "positive_probability": round(positive_probability, 4),
        "negative_probability": round(1 - positive_probability, 4),
        "confidence": round(confidence, 4)
    }

# Try your own review here:
my_review = "A wonderful movie with a touching story and excellent acting."
prediction = predict_sentiment(my_review)
display(pd.DataFrame([prediction]))

## 14. Save the trained model

In [ ]:
# Save the complete Keras model, including the TextVectorization layer.
MODEL_PATH = "imdb_sentiment_classifier.keras"
model.save(MODEL_PATH)

print("Model saved to:", MODEL_PATH)

# Optional: download the trained model from Colab.
from google.colab import files
files.download(MODEL_PATH)

## 15. Project conclusion

This project developed a binary sentiment-analysis classifier using TensorFlow/Keras and the IMDB movie-review dataset. The workflow included text decoding, tokenization and vectorization, word embeddings, a multi-layer neural network, Batch Normalization, Dropout, and Early Stopping. Training and validation convergence curves were plotted, and the final model was evaluated on an unseen test set. New review samples were passed to the model to produce sentiment labels and confidence estimates.

### Key learning outcomes
- Understanding an end-to-end NLP classification pipeline.
- Converting text into numerical token sequences.
- Learning word embeddings through a neural network.
- Applying regularization and Batch Normalization.
- Monitoring validation performance and stopping training early.
- Evaluating classification results and interpreting confidence scores.
- Saving a trained model for later reuse.

### Limitations
1. The model is trained on movie reviews and may not generalize to other text domains.
2. Sentiment can be ambiguous, sarcastic, mixed, or context-dependent.
3. The 0.5 threshold is a simple default and may not be optimal for every use.
4. Confidence scores are not necessarily calibrated probabilities.
5. Dataset biases and vocabulary limitations can affect predictions.

### Future improvements
- Compare with an LSTM, BiLSTM, or Transformer model.
- Tune vocabulary size, sequence length, embedding dimension, and dropout.
- Add cross-validation and calibration analysis.
- Test on a separate domain-specific dataset.
- Deploy the model with a simple API or web interface.

---
**End of project notebook**

## 16. Submission checklist

Before submitting:
- [ ] Run every notebook cell from top to bottom in Google Colab.
- [ ] Confirm the dataset downloads successfully.
- [ ] Ensure training and validation loss/accuracy plots are visible.
- [ ] Ensure test metrics and confusion matrix are visible.
- [ ] Ensure sample inference outputs are visible.
- [ ] Save a copy of the executed notebook with outputs.
- [ ] Include your name, course, institution, and date on the cover page if required by the company.
- [ ] Do not report expected or assumed accuracy; use the actual values generated when you run the notebook.